# 初始化模型的构建

一方面，许多主流的多因子模型当中都会考虑市值效应，另一方面，在先前的实证研究当中市值确实对策略的收益率起到了至关重要的作用，于是在构建初始化模型的时候首先考虑将市值效应加入进来，接下来我们要考虑使用什么样的代理变量来表征市值效应的暴露。在初始的尝试中我们采用较为常规的对数市值来作为市值效应的代理变量。将只考虑市值效应的模型作为M0，并且固定本次研究的股票池、训练期、测试期和训练标签，后面研究的模型完全采用同样的环境参数来比较。

### 固定研究环境

In [1]:
# Notebook 单元格 1：固定研究环境。后续模型沿用 STUDY，不在这里放模型参数。
import math
import time
import numpy as np
import pandas as pd
from IPython.display import display
from factor_lib.common.data_adapters.bigquant_adapters.loader import (
    load_trading_dates, load_factor_raw_data,
)
from factor_lib.common.data_adapters.bigquant_adapters.daily import load_daily_raw_data
from factor_lib.factor_hub.get_factor import get_factor

STUDY = {
    "universe": {"type": "all_a"},
    "train_start": "2020-01-01", "train_end": "2023-12-31",
    "test_start": "2024-01-01", "test_end": "2026-06-30",
    "rebalance_trading_days": 20,  # 也作为未来收益标签的交易日跨度
    "min_cross_section": 100,
    "top_fraction": 0.10,
    "hac_lags": None,
}


ModuleNotFoundError: No module named 'factor_lib'

### 定义模型评价函数

In [ ]:
# Notebook 单元格 2：只评价模型输出的分数，不负责训练或计算因子。
def evaluate_return_model(model_name, scores, labels, study, universe_panel=None,
                          show_progress=True, progress_every=20):
    """返回测试期汇总指标和逐信号日指标。分数越高，预测收益越高。"""
    if study["universe"]["type"] != "all_a" and universe_panel is None:
        raise ValueError("自定义/指数股票池必须提供逐信号日的 universe_panel。")
    scores = scores[["date", "instrument", "score"]].copy()
    labels = labels[["date", "instrument", "return_end_date", "forward_return"]].copy()
    for frame in (scores, labels):
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        if frame.duplicated(["date", "instrument"]).any():
            raise ValueError("模型分数或收益标签存在重复的 date + instrument。")
    scores["score"] = pd.to_numeric(scores["score"], errors="coerce")
    labels["forward_return"] = pd.to_numeric(labels["forward_return"], errors="coerce")
    labels["return_end_date"] = pd.to_datetime(labels["return_end_date"]).dt.normalize()
    if labels["return_end_date"].le(labels["date"]).any():
        raise ValueError("收益标签必须在信号日之后结束。")
    if universe_panel is not None:
        scores = scores.merge(universe_panel, on=["date", "instrument"],
                              validate="one_to_one")
        labels = labels.merge(universe_panel, on=["date", "instrument"],
                              validate="one_to_one")
    panel = scores.merge(labels, on=["date", "instrument"], validate="one_to_one")
    panel = panel.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["score", "forward_return"]
    )

    rows = []
    groups = list(panel.groupby("date", sort=True))
    started = time.perf_counter()
    for k, (date, section) in enumerate(groups, 1):
        if len(section) < study["min_cross_section"] or section["score"].nunique() < 2:
            raise ValueError(f"{date:%Y-%m-%d} 的有效股票或分数变化不足。")
        n_top = math.ceil(len(section) * study["top_fraction"])
        top = section.sort_values(
            ["score", "instrument"], ascending=[False, True], kind="mergesort"
        ).head(n_top)
        rows.append({
            "date": date,
            "return_end_date": section["return_end_date"].iloc[0],
            "n_stocks": len(section),
            "rank_ic": section["score"].corr(
                section["forward_return"], method="spearman"
            ),
            "top_excess_gross": top["forward_return"].mean()
                                - section["forward_return"].mean(),
        })
        if show_progress and (k % progress_every == 0 or k == len(groups)):
            print(f"\r[模型评价] {k}/{len(groups)}，{date:%Y-%m-%d}，"
                  f"已耗时 {time.perf_counter() - started:.1f}s", end="", flush=True)
    if show_progress:
        print()
    per_date = pd.DataFrame(rows)
    if per_date.empty:
        raise ValueError("测试期没有完整的评价截面。")

    def nw_t(values):
        x = pd.to_numeric(values, errors="coerce").dropna().to_numpy(float)
        n = len(x)
        if n < 2:
            return np.nan
        lags = study["hac_lags"]
        lag_count = min(n - 1, lags if lags is not None
                        else math.floor(4 * (n / 100) ** (2 / 9)))
        centered = x - x.mean()
        long_run_var = centered @ centered / n
        for lag in range(1, lag_count + 1):
            long_run_var += 2 * (1 - lag / (lag_count + 1)) * (
                centered[lag:] @ centered[:-lag] / n
            )
        se = math.sqrt(max(long_run_var, 0) / n)
        return x.mean() / se if se > 0 else np.nan

    summary = pd.DataFrame([{
        "model": model_name,
        "test_dates": len(per_date),
        "mean_stocks": per_date["n_stocks"].mean(),
        "rank_ic_mean": per_date["rank_ic"].mean(),
        "rank_ic_nw_t": nw_t(per_date["rank_ic"]),
        "top_excess_mean_gross": per_date["top_excess_gross"].mean(),
        "top_excess_nw_t": nw_t(per_date["top_excess_gross"]),
    }])
    return summary, per_date


### 标签数据准备

In [ ]:
# Notebook 单元格 3：按固定研究环境准备一次共享标签，后续模型直接复用。
calendar = load_trading_dates(STUDY["train_start"], STUDY["test_end"])
horizon = STUDY["rebalance_trading_days"]
schedule_rows = []
for split in ("train", "test"):
    start = pd.Timestamp(STUDY[f"{split}_start"])
    end = pd.Timestamp(STUDY[f"{split}_end"])
    dates = calendar[(calendar >= start) & (calendar <= end)]
    for date in dates[::horizon]:
        end_position = calendar.get_loc(date) + horizon
        if end_position < len(calendar) and calendar[end_position] <= end:
            schedule_rows.append((split, date, calendar[end_position]))
schedule = pd.DataFrame(schedule_rows, columns=["split", "date", "return_end_date"])
if not {"train", "test"}.issubset(set(schedule["split"])):
    raise ValueError("训练期或测试期没有完整的未来收益标签。")
signal_dates = pd.DatetimeIndex(schedule["date"].unique())

# 股票池在信号日确定；指数股票池读取当日历史成分。
universe = STUDY["universe"]
universe_panel, instruments = None, None
if universe["type"] == "custom":
    instruments = list(universe["instruments"])
    universe_panel = pd.MultiIndex.from_product(
        [signal_dates, instruments], names=["date", "instrument"]
    ).to_frame(index=False)
elif universe["type"] == "index":
    import dai
    index_codes = ", ".join("'" + code.replace("'", "''") + "'"
                            for code in universe["index_codes"])
    date_codes = ", ".join("'" + date.strftime("%Y-%m-%d") + "'"
                           for date in signal_dates)
    universe_panel = dai.query(
        f"SELECT date, member_code AS instrument FROM cn_stock_index_component "
        f"WHERE instrument IN ({index_codes}) AND date IN ({date_codes})",
        filters={"date": [signal_dates.min().strftime("%Y-%m-%d"),
                          signal_dates.max().strftime("%Y-%m-%d")]},
    ).df()
    universe_panel["date"] = pd.to_datetime(universe_panel["date"]).dt.normalize()
    universe_panel = universe_panel.drop_duplicates(["date", "instrument"])
    if set(signal_dates) - set(universe_panel["date"]):
        raise ValueError("部分信号日没有查询到指数历史成分股。")
    instruments = universe_panel["instrument"].unique().tolist()
elif universe["type"] != "all_a":
    raise ValueError("universe.type 仅支持 all_a、custom、index。")

# 标签为信号日收盘至未来 horizon 个交易日收盘的后复权收益。
price_dates = sorted(set(schedule["date"]) | set(schedule["return_end_date"]))
prices = load_daily_raw_data(
    ["close"], dates=price_dates, instruments=instruments, show_progress=True,
)
prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()
prices["close"] = pd.to_numeric(prices["close"], errors="coerce")
prices = prices.loc[prices["close"].gt(0)]
if prices.duplicated(["date", "instrument"]).any():
    raise ValueError("价格存在重复的 date + instrument。")
labels = schedule.merge(
    prices.rename(columns={"close": "start_close"}),
    on="date", validate="one_to_many",
).merge(
    prices.rename(columns={"date": "return_end_date", "close": "end_close"}),
    on=["return_end_date", "instrument"], validate="one_to_one",
)
labels["forward_return"] = labels["end_close"] / labels["start_close"] - 1
labels = labels[["split", "date", "instrument", "return_end_date",
                 "forward_return"]]
if universe_panel is not None:
    labels = labels.merge(universe_panel, on=["date", "instrument"],
                          validate="one_to_one")

train_labels = labels.loc[labels["split"].eq("train")].copy()
test_labels = labels.loc[labels["split"].eq("test")].copy()
if train_labels.empty or test_labels.empty:
    raise ValueError("训练期或测试期标签为空，请检查日期、股票池和价格覆盖。")
print(f"共享标签准备完成：训练期 {len(train_labels):,} 条，"
      f"测试期 {len(test_labels):,} 条；后续模型无需重复查询标签价格。")


### 构建M0

In [ ]:
# Notebook 单元格 4：M0 特征、训练期拟合与系数检验；直接复用共享标签。
MODEL = {"name": "M0", "factor_name": "log_market_cap"}
raw = load_factor_raw_data(
    MODEL["factor_name"], dates=signal_dates, factor_params={},
    instruments=instruments, show_progress=True,
)
factor = get_factor(
    MODEL["factor_name"], raw, target_dates=signal_dates,
    as_of_date=STUDY["test_end"], show_progress=True,
)
factor["date"] = pd.to_datetime(factor["date"]).dt.normalize()
if universe_panel is not None:
    factor = factor.merge(universe_panel, on=["date", "instrument"],
                          validate="one_to_one")
factor[MODEL["factor_name"]] = pd.to_numeric(
    factor[MODEL["factor_name"]], errors="coerce"
)
factor["size_z"] = factor.groupby("date")[MODEL["factor_name"]].transform(
    lambda values: (values - values.mean()) / values.std(ddof=0)
    if values.std(ddof=0) > 0 else np.nan
)

# 只将训练期特征与 train_labels 配对，逐期估计市值斜率。
train_dates = schedule.loc[schedule["split"].eq("train"), "date"]
train_panel = factor.loc[factor["date"].isin(train_dates)].merge(
    train_labels, on=["date", "instrument"], validate="one_to_one",
).dropna(subset=["size_z", "forward_return"])
slopes = []
for date, section in train_panel.groupby("date", sort=True):
    if len(section) < STUDY["min_cross_section"]:
        raise ValueError(f"训练截面 {date:%Y-%m-%d} 的样本不足。")
    x = section["size_z"].to_numpy(float)
    y = section["forward_return"].to_numpy(float)
    if np.std(x) == 0:
        raise ValueError(f"训练截面 {date:%Y-%m-%d} 的规模暴露无差异。")
    slopes.append(np.linalg.lstsq(
        np.column_stack([np.ones(len(x)), x]), y, rcond=None
    )[0][1])
beta = float(np.mean(slopes))
if not np.isfinite(beta) or beta == 0:
    raise ValueError("M0 的训练期系数无效。")

# 训练期系数的 Newey–West 标准误和近似双侧显著性检验。
train_slopes = np.asarray(slopes, dtype=float)
n_periods = len(train_slopes)
if n_periods < 2:
    raise ValueError("训练期有效截面不足两个，无法检验系数显著性。")
lag_setting = STUDY["hac_lags"]
if lag_setting is not None and lag_setting < 0:
    raise ValueError("hac_lags 必须是非负整数或 None。")
lag_count = min(
    n_periods - 1,
    int(lag_setting) if lag_setting is not None
    else math.floor(4 * (n_periods / 100) ** (2 / 9)),
)
centered = train_slopes - beta
long_run_var = centered @ centered / n_periods
for lag in range(1, lag_count + 1):
    long_run_var += 2 * (1 - lag / (lag_count + 1)) * (
        centered[lag:] @ centered[:-lag] / n_periods
    )
beta_hac_se = math.sqrt(max(float(long_run_var), 0.0) / n_periods)
beta_hac_t = beta / beta_hac_se if beta_hac_se > 0 else np.nan
beta_hac_p = math.erfc(abs(beta_hac_t) / math.sqrt(2)) if np.isfinite(beta_hac_t) else np.nan
m0_coefficient_inference = pd.DataFrame([{
    "variable": MODEL["factor_name"],
    "coefficient_mean": beta,
    "hac_standard_error": beta_hac_se,
    "hac_t": beta_hac_t,
    "two_sided_p_approx": beta_hac_p,
    "significant_at_5pct": bool(beta_hac_p < 0.05) if np.isfinite(beta_hac_p) else False,
    "n_train_dates": n_periods,
    "hac_lags": lag_count,
}])

# 测试期只用当日特征和训练得到的 beta 形成分数，不使用 test_labels。
test_dates = schedule.loc[schedule["split"].eq("test"), "date"]
m0_scores = factor.loc[
    factor["date"].isin(test_dates), ["date", "instrument", "size_z"]
].rename(columns={"size_z": "score"})
m0_scores["score"] = beta * m0_scores["score"]
print(f"M0 已构建：训练期斜率均值 {beta:.6g}；测试期 {len(test_dates)} 个信号日。")
display(m0_coefficient_inference)


### 评价M0

In [ ]:
# Notebook 单元格 5：固定评价函数只接收 M0 测试期分数及共享测试期标签。
# 标签是信号日收盘至未来 horizon 日收盘的后复权收益，仅评价预测排序；
# 不代表可成交或扣除交易成本后的策略收益。
m0_evaluation, m0_per_date = evaluate_return_model(
    MODEL["name"], m0_scores, test_labels, STUDY,
    universe_panel=universe_panel,
)
m0_evaluation.insert(1, "train_beta", beta)
display(m0_evaluation)
